You are working with two complementary datasets describing a large-scale biotechnological production process.

The first dataset (“4000 series operating data”) contains high-frequency sensor measurements collected every 5–15 minutes over several hundred hours for 22 individual batches. It includes 17 process variables such as liquid inflows, gas inflows, pH, off-gas measurements, pressure, and oxygen levels. In total, the dataset contains approximately 1.4 million time-stamped observations. Some values are missing due to sensor limitations or non-operational readings, but all batches are considered representative of normal process operation.

The second dataset contains laboratory measurements recorded approximately every 4 hours for 21 batches. This dataset includes the product concentration values over time, which are used to calculate a yield parameter for each batch. By combining the time-series sensor data with the laboratory product data, it is possible to construct a data-driven model to identify which process variables most strongly influence product yield and to predict the yield of a previously unseen batch.

The operating dataset contains 83,204 time-series observations across 18 columns. Each row corresponds to a timestamped measurement for a specific batch. The Batch column is an integer identifier, Date and time is stored as a datetime variable, and all process variables (LIQUID, GAS, OFFGAS, PRESSURE, pH, and OXYGEN) are continuous numerical features stored as float64. Some missing values are present due to sensor downtime or non-operational readings.

To reduce dimensionality and improve interpretability, related sensor streams were aggregated. The six liquid inflow variables were summed into TOTAL_LIQUID_INFLOW, the four gas inflow variables into TOTAL_GAS_INFLOW, the two off-gas sensors were averaged into TOTAL_OFFGAS, and the two pressure sensors were averaged into MEAN_PRESSURE. This transformation reduces redundancy from multiple sensors measuring similar physical phenomena, mitigates noise between parallel sensors, and simplifies the feature space while preserving the key process information relevant for yield prediction.

Feature Aggregation and Dimensionality Reduction

The operating dataset contains multiple sensor measurements representing similar physical quantities recorded from parallel inflow streams and duplicate sensors. To improve interpretability and reduce redundancy, related variables were aggregated into composite features.

The six liquid inflow variables (LIQUID, LIQUID.1, LIQUID.2, LIQUID.3, LIQUID.4, LIQUID.5) represent separate feed streams entering the same bioreactor. Since the overall system behaviour depends on the total liquid entering the reactor, these variables were summed to create a single feature, TOTAL_LIQUID_INFLOW. Similarly, the four gas inflow variables (GAS, GAS.1, GAS.2, GAS.3) were summed to form TOTAL_GAS_INFLOW, representing the total gas supply to the system.

In contrast, the OFFGAS and PRESSURE variables represent duplicate sensors measuring the same physical process conditions. These were averaged to create TOTAL_OFFGAS and MEAN_PRESSURE, respectively. Averaging improves robustness by reducing measurement noise and mitigating small calibration differences between sensors, while preserving the true underlying process state.

Following aggregation, the original individual sensor columns were removed to reduce dimensionality and multicollinearity. This simplification improves model stability, reduces the risk of overfitting, and ensures that the features more directly reflect the underlying physical processes influencing product yield.

Import data

In [2]:
from applied.data_processing import (
    load_operating_data,
    load_product_data,
    prepare_operating_timeseries,
    batch_time_diagnostics,
    build_features_and_target
)

In [3]:
from pathlib import Path

# Project root = one level above notebooks/
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

operating_csv = DATA_DIR / "4000 series operating data.csv"
product_xlsx = DATA_DIR / "4000 series product data.xlsx"

op_df = load_operating_data(operating_csv)
prod_df = load_product_data(product_xlsx)

op_df = prepare_operating_timeseries(
    op_df,
    add_tier=False,
    resample_rule="1h"
)

#need to add denosier and standazing - r we meant to standize before comparing timeseries ?? since we will stadize different depdning on val set ? 
#need to make index to be able to overlay timeseries and not have them span the year 
op_df


pH      OXYGEN  \
productivity_rank Batch Date and time                               
1                 4041  2020-03-18 09:00:00  6.165000    8.820000   
                        2020-03-18 10:00:00  5.897500    7.090000   
                        2020-03-18 11:00:00  5.787500    8.435000   
                        2020-03-18 12:00:00  5.795000   38.120000   
                        2020-03-18 13:00:00  5.802500    8.602500   
...                                               ...         ...   
22                4053  2021-05-25 17:00:00  5.801667   13.085000   
                        2021-05-25 18:00:00  5.791667   10.825000   
                        2021-05-25 19:00:00  5.803333   12.715833   
                        2021-05-25 20:00:00  5.870833   68.421667   
                        2021-05-25 21:00:00  5.760000  111.818750   

                                             TOTAL_LIQUID_INFLOW  \
productivity_rank Batch Date and time                              
1                 4041  2020-03-18 09:00:00         15994.080000   
                        2020-03-18 10:00:00         16277.535000   
                        2020-03-18 11:00:00         18091.965000   
                        2020-03-18 12:00:00         18324.007500   
                        2020-03-18 13:00:00         18315.392500   
...                                                          ...   
22                4053  2021-05-25 17:00:00         24041.537500   
                        2021-05-25 18:00:00         23967.733333   
                        2021-05-25 19:00:00         23774.552500   
                        2021-05-25 20:00:00         23788.894167   
                        2021-05-25 21:00:00         21133.936250   

                                             TOTAL_GAS_INFLOW  MEAN_OFFGAS  \
productivity_rank Batch Date and time                                        
1                 4041  2020-03-18 09:00:00       7586.485000    28.050000   
                        2020-03-18 10:00:00       7624.742500    29.657500   
                        2020-03-18 11:00:00       8019.370000    33.430000   
                        2020-03-18 12:00:00       8131.182500    31.398750   
                        2020-03-18 13:00:00       8169.685000    18.198750   
...                                                       ...          ...   
22                4053  2021-05-25 17:00:00      12223.495833     3.422083   
                        2021-05-25 18:00:00      12232.390833     3.423333   
                        2021-05-25 19:00:00      12229.294167     3.429167   
                        2021-05-25 20:00:00      12187.012500     3.042083   
                        2021-05-25 21:00:00      11354.962500     2.587500   

                                             MEAN_PRESSURE  
productivity_rank Batch Date and time                       
1                 4041  2020-03-18 09:00:00       3.357500  
                        2020-03-18 10:00:00       3.413750  
                        2020-03-18 11:00:00       3.541250  
                        2020-03-18 12:00:00       3.558750  
                        2020-03-18 13:00:00       3.628750  
...                                                    ...  
22                4053  2021-05-25 17:00:00       3.636250  
                        2021-05-25 18:00:00       3.633750  
                        2021-05-25 19:00:00       3.608333  
                        2021-05-25 20:00:00       3.591667  
                        2021-05-25 21:00:00       3.546875  

[14393 rows x 6 columns]

In [4]:
diag = batch_time_diagnostics(op_df)

diag

,,Start,End,Duration_hours,Duration_days,Calendar_days_spanned,Expected_samples,Actual_samples,Missing_samples,Percent_missing
productivity_rank,Batch,,,,,,,,,
1,4041,2020-03-18 09:00:00,2020-04-15 23:00:00,686.0,28.583333,29,687,687,0,0.0
2,4043,2020-06-09 19:00:00,2020-06-28 16:00:00,453.0,18.875000,20,454,454,0,0.0
3,4047,2020-10-13 23:00:00,2020-11-13 20:00:00,741.0,30.875000,32,742,742,0,0.0
4,4040,2020-02-08 11:00:00,2020-03-08 04:00:00,689.0,28.708333,30,690,690,0,0.0
5,4042,2020-04-24 00:00:00,2020-05-29 23:00:00,863.0,35.958333,36,864,864,0,0.0
6,4046,2020-08-28 01:00:00,2020-10-04 13:00:00,900.0,37.500000,38,901,901,0,0.0
7,4045,2020-07-28 20:00:00,2020-08-11 16:00:00,332.0,13.833333,15,333,333,0,0.0
8,4052,2021-03-10 21:00:00,2021-04-13 01:00:00,796.0,33.166667,35,797,797,0,0.0
9,4034,2019-07-02 13:00:00,2019-08-03 23:00:00,778.0,32.416667,33,779,779,0,0.0


In [ ]:
import dearpygui.dearpygui as dpg
import pandas as pd

# -----------------------
# Prepare Data
# -----------------------

df_plot = op_df.reset_index()

# Convert datetime to numeric for DearPyGui
df_plot["timestamp"] = df_plot["Date and time"].astype("int64") // 10**9

# -----------------------
# DearPyGui Setup
# -----------------------

dpg.create_context()

with dpg.window(label="Hourly Total Liquid Inflow (All Batches)", width=1000, height=600):

    with dpg.plot(label="Hourly Inflow by Batch", height=-1, width=-1):

        dpg.add_plot_legend()

        x_axis = dpg.add_plot_axis(dpg.mvXAxis, label="Time (Unix)")
        y_axis = dpg.add_plot_axis(dpg.mvYAxis, label="TOTAL_LIQUID_INFLOW")

        # -----------------------
        # Loop Through Batches
        # -----------------------

        for batch_id, batch_data in df_plot.groupby("Batch"):

            batch_data = batch_data.sort_values("Date and time")

            x_data = batch_data["timestamp"].tolist()
            y_data = batch_data["TOTAL_LIQUID_INFLOW"].tolist()

            dpg.add_line_series(
                x_data,
                y_data,
                label=f"Batch {int(batch_id)}",
                parent=y_axis
            )

# -----------------------
# Launch
# -----------------------

dpg.create_viewport(title="Hourly Batch Plot", width=1200, height=700)
dpg.setup_dearpygui()
dpg.show_viewport()
dpg.start_dearpygui()
dpg.destroy_context()


: 